In [ ]:
# === NORTHSTAR -- Step [0] of Pipeline ===
# Tower 9: Tower of the World -- i18n QA for solace-browser
# Every probe checks REAL code. No mocks, no fakes.

NORTHSTAR = "world-i18n-qa"
NOTEBOOK_PATH = "notebooks/qa/04-world-i18n-qa.ipynb"
TOWER = 9
TOWER_NAME = "Tower of the World"
TOTAL_PROBES = 17
MIN_SCORE = 70

# Tower DNA: global(reach) = locale(correct) x culture(respected) x script(rendered) x format(native)
# 49 floors, 343 questions, target: solace-browser

import os
import json
import re
import glob as globmod
from pathlib import Path

PROJECT_ROOT = "/home/phuc/projects/solace-browser"
WEB_DIR = os.path.join(PROJECT_ROOT, "web")
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
CSS_FILE = os.path.join(WEB_DIR, "css", "site.css")
LOCALES_DIR = os.path.join(PROJECT_ROOT, "app", "locales", "yinyang")
I18N_FILE = os.path.join(SRC_DIR, "i18n.py")
SETTINGS_FILE = os.path.join(SRC_DIR, "settings_manager.py")

HTML_FILES = [
    "home.html", "settings.html", "app-store.html", "download.html",
    "docs.html", "demo.html", "start.html", "schedule.html",
    "tunnel-connect.html", "machine-dashboard.html", "style-guide.html",
    "create-app.html", "app-detail.html",
    "partials-header.html", "partials-footer.html",
    "docs/quick-start.html", "docs/oauth3.html", "docs/mcp.html",
]

JS_FILES = [
    "js/layout.js", "js/solace.js", "js/yinyang-rail.js",
    "js/yinyang-tutorial.js", "js/yinyang-tutorial-v2.js",
    "js/yinyang-delight.js", "js/yinyang-oauth3-confirm.js",
    "js/onboarding.js", "js/schedule-core.js",
    "js/schedule-approvals.js", "js/schedule-evidence.js",
    "js/schedule-calendar.js",
]

# Expected locales from i18n.py _SUPPORTED set
EXPECTED_LOCALES = {
    "en", "es", "vi", "zh", "pt", "fr", "ja", "de", "ar", "hi", "ko", "id", "ru",
    "zh-hant", "tr", "pl", "th", "nl", "it", "uk", "sv", "he", "fa",
}

RTL_LOCALES = {"ar", "he", "fa"}

results = []  # (probe_name, pass_bool, detail)

print(f"NORTHSTAR: {NORTHSTAR}")
print(f"TOWER: {TOWER} -- {TOWER_NAME}")
print(f"PROBES: {TOTAL_PROBES}")
print(f"MIN_SCORE: {MIN_SCORE}%")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"HTML surfaces: {len(HTML_FILES)} files")
print(f"JS files: {len(JS_FILES)} files")
print(f"Expected locales: {len(EXPECTED_LOCALES)}")

In [ ]:
# === Cell 1: Bootstrap -- helper functions ===

def read_file(path):
    """Read file content as string, return empty string if missing."""
    try:
        return Path(path).read_text(encoding="utf-8")
    except (FileNotFoundError, UnicodeDecodeError):
        return ""

def read_json(path):
    """Read and parse JSON file, return empty dict if missing or invalid."""
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError, UnicodeDecodeError):
        return {}

def find_py_files(directory):
    """Recursively find all .py files under directory."""
    return list(Path(directory).rglob("*.py"))

def record(probe_name, passed, detail=""):
    """Record a probe result and print PASS/FAIL."""
    results.append((probe_name, passed, detail))
    tag = "PASS" if passed else "FAIL"
    print(f"{tag}: {probe_name}")
    if detail:
        print(f"  {detail}")

print("Bootstrap loaded.")

In [ ]:
# === Probe 1: Locale file existence ===
# Floors 1-13 (existing locales): Do JSON files exist for all _SUPPORTED locales?
# Tower Q: "Is the product fully functional with no untranslated strings?"

existing_files = set()
missing_files = []
for loc in sorted(EXPECTED_LOCALES):
    path = os.path.join(LOCALES_DIR, f"{loc}.json")
    if os.path.isfile(path):
        existing_files.add(loc)
    else:
        missing_files.append(loc)

# Also check for unexpected locale files on disk
on_disk = set()
if os.path.isdir(LOCALES_DIR):
    for f in os.listdir(LOCALES_DIR):
        if f.endswith(".json"):
            on_disk.add(f.replace(".json", ""))

extra = on_disk - EXPECTED_LOCALES
coverage = len(existing_files) / len(EXPECTED_LOCALES) * 100 if EXPECTED_LOCALES else 0

record(
    "locale-files-exist",
    len(missing_files) == 0,
    f"{len(existing_files)}/{len(EXPECTED_LOCALES)} locale files present ({coverage:.0f}%). "
    f"Missing: {missing_files or 'none'}. Extra on disk: {extra or 'none'}"
)

In [ ]:
# === Probe 2: Locale key completeness ===
# Floors 1-13: Do non-English locales have the same keys as English?
# Tower Q: "Does the [locale] extend beyond the homepage into recipe descriptions?"

def flatten_keys(d, prefix=""):
    """Flatten nested dict to dot-separated keys."""
    keys = set()
    for k, v in d.items():
        if k.startswith("_"):
            continue
        full = f"{prefix}{k}" if prefix == "" else f"{prefix}.{k}"
        if isinstance(v, dict):
            keys.update(flatten_keys(v, full))
        else:
            keys.add(full)
    return keys

en_data = read_json(os.path.join(LOCALES_DIR, "en.json"))
en_keys = flatten_keys(en_data)
print(f"English locale has {len(en_keys)} keys\n")

incomplete_locales = []
for loc in sorted(existing_files - {"en"}):
    loc_data = read_json(os.path.join(LOCALES_DIR, f"{loc}.json"))
    loc_keys = flatten_keys(loc_data)
    missing = en_keys - loc_keys
    pct = len(loc_keys & en_keys) / len(en_keys) * 100 if en_keys else 0
    if missing:
        incomplete_locales.append((loc, len(missing), pct))
        print(f"  {loc}: {pct:.0f}% complete, missing {len(missing)} keys")

all_complete = len(incomplete_locales) == 0
record(
    "locale-key-completeness",
    all_complete,
    f"{len(existing_files) - 1 - len(incomplete_locales)}/{len(existing_files) - 1} non-English locales fully complete. "
    f"Incomplete: {len(incomplete_locales)}"
)

In [ ]:
# === Probe 3: Cross-script contamination in locale files ===
# Floor 10 (Korean 15.7% contamination), Floor 13 (Arabic 30+ contaminated strings)
# Tower Q: "What percentage of Korean strings contain characters from other scripts?"

import unicodedata

def get_script(char):
    """Return the Unicode script name for a character."""
    try:
        name = unicodedata.name(char, "")
    except ValueError:
        return "UNKNOWN"
    if "CJK" in name or "IDEOGRAPH" in name:
        return "CJK"
    if "HANGUL" in name:
        return "HANGUL"
    if "HIRAGANA" in name or "KATAKANA" in name:
        return "JAPANESE"
    if "ARABIC" in name:
        return "ARABIC"
    if "CYRILLIC" in name:
        return "CYRILLIC"
    if "DEVANAGARI" in name:
        return "DEVANAGARI"
    if "THAI" in name:
        return "THAI"
    if "LATIN" in name or char.isascii():
        return "LATIN"
    if "HEBREW" in name:
        return "HEBREW"
    return "OTHER"

# Define expected primary scripts per locale
LOCALE_SCRIPTS = {
    "ko": {"HANGUL", "LATIN"},
    "ar": {"ARABIC", "LATIN"},
    "he": {"HEBREW", "LATIN"},
    "fa": {"ARABIC", "LATIN"},
    "hi": {"DEVANAGARI", "LATIN"},
    "ja": {"CJK", "JAPANESE", "LATIN"},
    "zh": {"CJK", "LATIN"},
    "zh-hant": {"CJK", "LATIN"},
    "ru": {"CYRILLIC", "LATIN"},
    "uk": {"CYRILLIC", "LATIN"},
    "th": {"THAI", "LATIN"},
    "vi": {"LATIN"},
}

contaminated_locales = []
for loc, allowed in sorted(LOCALE_SCRIPTS.items()):
    path = os.path.join(LOCALES_DIR, f"{loc}.json")
    data = read_json(path)
    if not data:
        continue

    loc_keys = flatten_keys(data)
    counts = {"total": 0, "contaminated": 0}
    foreign_scripts = set()

    def check_val(val, counts=counts, allowed=allowed, foreign_scripts=foreign_scripts):
        if isinstance(val, str):
            counts["total"] += 1
            for ch in val:
                if ch.isspace() or ch in '{}()[]<>.,;:!?"\'-/\\@#$%^&*+=_~`|0123456789':
                    continue
                sc = get_script(ch)
                if sc not in allowed and sc != "OTHER":
                    counts["contaminated"] += 1
                    foreign_scripts.add(sc)
                    break
        elif isinstance(val, dict):
            for v in val.values():
                check_val(v, counts, allowed, foreign_scripts)
        elif isinstance(val, list):
            for v in val:
                check_val(v, counts, allowed, foreign_scripts)

    check_val(data)
    pct = (counts["contaminated"] / counts["total"] * 100) if counts["total"] > 0 else 0
    if pct > 5:
        contaminated_locales.append(f"{loc}: {pct:.1f}% ({counts['contaminated']}/{counts['total']})")

cross_script_ok = len(contaminated_locales) == 0
record(
    "cross-script-contamination",
    cross_script_ok,
    f"Contaminated locales (>5%): {contaminated_locales}" if contaminated_locales else "All locales clean"
)

In [ ]:
# === Probe 4: Hardcoded English text in HTML ===
# Floor 1 (James): "Are all error messages, help text, and recipe descriptions complete?"
# Floor 49 (Ambassador): "Can a monolingual speaker use Solace without encountering foreign characters?"
# This checks: are there user-visible English strings hardcoded in HTML instead of using i18n keys?

# Pattern: look for text content in HTML that is NOT inside a data-i18n attribute,
# script tag, or style tag, and contains common English UI words
HARDCODED_PATTERNS = [
    r'>\s*(Submit|Cancel|Delete|Save|Edit|Close|Open|Loading|Error|Success|Warning|Confirm)\s*<',
    r'>\s*(Sign [Ii]n|Sign [Uu]p|Log [Ii]n|Log [Oo]ut|Reset|Continue)\s*<',
    r'>\s*(Settings|Dashboard|Profile|Account|Help|About|Contact)\s*<',
]

# Regex to strip <script> and <style> blocks before scanning
STRIP_RE = re.compile(r'<(script|style)[^>]*>.*?</\1>', re.DOTALL | re.IGNORECASE)

hardcoded_hits = []
for html_file in HTML_FILES:
    path = os.path.join(WEB_DIR, html_file)
    content = read_file(path)
    if not content:
        continue
    # Strip script/style
    stripped = STRIP_RE.sub("", content)
    for pattern in HARDCODED_PATTERNS:
        matches = re.findall(pattern, stripped)
        for m in matches:
            hardcoded_hits.append((html_file, m))

# Check if data-i18n attributes are used anywhere
data_i18n_count = 0
for html_file in HTML_FILES:
    path = os.path.join(WEB_DIR, html_file)
    content = read_file(path)
    data_i18n_count += len(re.findall(r'data-i18n', content))

# Also check for JS-based i18n injection (window.YINYANG_I18N)
js_i18n_refs = 0
for js_file in JS_FILES:
    path = os.path.join(WEB_DIR, js_file)
    content = read_file(path)
    js_i18n_refs += len(re.findall(r'YINYANG_I18N|window\.YINYANG_I18N', content))

if hardcoded_hits:
    for (f, m) in hardcoded_hits[:10]:
        print(f"  {f}: hardcoded '{m}'")
    if len(hardcoded_hits) > 10:
        print(f"  ... and {len(hardcoded_hits) - 10} more")

print(f"\n  data-i18n attributes in HTML: {data_i18n_count}")
print(f"  JS references to YINYANG_I18N: {js_i18n_refs}")

# Pass if few hardcoded strings AND some i18n mechanism exists
has_i18n_mechanism = data_i18n_count > 0 or js_i18n_refs > 0
record(
    "hardcoded-english-in-html",
    len(hardcoded_hits) <= 5 and has_i18n_mechanism,
    f"{len(hardcoded_hits)} hardcoded English strings found. "
    f"i18n mechanism: {'data-i18n=' + str(data_i18n_count) if data_i18n_count else ''} "
    f"{'JS_I18N=' + str(js_i18n_refs) if js_i18n_refs else ''}"
)

In [ ]:
# === Probe 5: RTL CSS rules exist ===
# Floor 13 (Fatima/Arabic): "Are the 170+ RTL CSS rules in solace-browser activated by any JavaScript code?"
# Floor 22 (Dariush/Persian): "Would the RTL dead code block Persian just as it blocks Arabic?"
# Floor 23 (Avi/Hebrew): "Does the RTL dead code block Hebrew?"

css_content = read_file(CSS_FILE)
rtl_rules = re.findall(r'\[dir="rtl"\]', css_content)
rtl_rule_count = len(rtl_rules)

# Check schedule.css too
sched_css = read_file(os.path.join(WEB_DIR, "css", "schedule.css"))
sched_rtl = len(re.findall(r'\[dir="rtl"\]', sched_css))

print(f"  site.css: {rtl_rule_count} [dir=\"rtl\"] selectors")
print(f"  schedule.css: {sched_rtl} [dir=\"rtl\"] selectors")

record(
    "rtl-css-rules-exist",
    rtl_rule_count >= 10,
    f"{rtl_rule_count + sched_rtl} total RTL CSS selectors across stylesheets"
)

In [ ]:
# === Probe 6: RTL activation in JavaScript ===
# Floor 13 (Fatima): "Is dir='rtl' set on the HTML element when Arabic locale is active?"
# Critical: RTL CSS is dead code if no JS sets dir="rtl" on <html>

rtl_activation_found = False
rtl_activation_files = []

for js_file in JS_FILES:
    path = os.path.join(WEB_DIR, js_file)
    content = read_file(path)
    if not content:
        continue
    # Look for RTL activation patterns
    patterns = [
        r'dir.*rtl',
        r'YINYANG_DIR',
        r'get_direction',
        r'is_rtl',
    ]
    for pat in patterns:
        if re.search(pat, content, re.IGNORECASE):
            rtl_activation_found = True
            rtl_activation_files.append((js_file, pat))

# Also check i18n.py for RTL support
i18n_content = read_file(I18N_FILE)
i18n_has_rtl = "_RTL_LOCALES" in i18n_content
i18n_has_direction = "get_direction" in i18n_content
i18n_has_js_bundle = "YINYANG_DIR" in i18n_content

# Check if partials-header.html has dir attribute logic
header_content = read_file(os.path.join(WEB_DIR, "partials-header.html"))
header_has_rtl = bool(re.search(r'dir=', header_content))

print(f"  i18n.py has _RTL_LOCALES: {i18n_has_rtl}")
print(f"  i18n.py has get_direction(): {i18n_has_direction}")
print(f"  i18n.py js_bundle() sets YINYANG_DIR: {i18n_has_js_bundle}")
print(f"  partials-header.html has dir=: {header_has_rtl}")
print(f"  JS files with RTL activation: {len(rtl_activation_files)}")
for (f, p) in rtl_activation_files:
    print(f"    {f}: matches '{p}'")

record(
    "rtl-js-activation",
    rtl_activation_found or (i18n_has_js_bundle and header_has_rtl),
    f"RTL activation: JS={rtl_activation_found}, i18n.py bundle={i18n_has_js_bundle}, "
    f"header dir={header_has_rtl}. "
    f"{'RTL CSS is LIVE' if (rtl_activation_found or header_has_rtl) else 'RTL CSS may be DEAD CODE'}"
)

In [ ]:
# === Probe 7: HTML lang attribute is dynamic ===
# Floor 49 (Ambassador): "In 2030 with 100 countries, does the current i18n infrastructure scale?"
# All HTML files hardcode lang="en" -- this must change dynamically per locale

hardcoded_lang_en = []
dynamic_lang = []

for html_file in HTML_FILES:
    path = os.path.join(WEB_DIR, html_file)
    content = read_file(path)
    if not content:
        continue
    if re.search(r'<html\s+lang="en"', content):
        hardcoded_lang_en.append(html_file)
    elif re.search(r'<html.*lang=', content):
        dynamic_lang.append(html_file)

print(f"  HTML files with hardcoded lang=\"en\": {len(hardcoded_lang_en)}/{len(HTML_FILES)}")
if hardcoded_lang_en:
    for f in hardcoded_lang_en[:5]:
        print(f"    {f}")
    if len(hardcoded_lang_en) > 5:
        print(f"    ... and {len(hardcoded_lang_en) - 5} more")

# Check if JS dynamically sets document.documentElement.lang
js_sets_lang = False
for js_file in JS_FILES:
    path = os.path.join(WEB_DIR, js_file)
    content = read_file(path)
    if re.search(r'documentElement\.lang|document\.documentElement\.setAttribute.*lang', content):
        js_sets_lang = True
        print(f"  JS dynamically sets lang: {js_file}")

record(
    "html-lang-attribute-dynamic",
    js_sets_lang or len(hardcoded_lang_en) == 0,
    f"{len(hardcoded_lang_en)} files hardcode lang=\"en\". "
    f"JS dynamically sets lang: {js_sets_lang}. "
    f"{'Screen readers will always announce English' if hardcoded_lang_en and not js_sets_lang else 'OK'}"
)

In [ ]:
# === Probe 8: Font stacks for CJK, Arabic, Devanagari ===
# Floor 12 (Yuki/Japanese): "Are Japanese web fonts declared or system fonts only?"
# Floor 13 (Fatima/Arabic): "Are Arabic font stacks declared in the CSS?"
# Floor 11 (Priya/Hindi): "Does Devanagari script render correctly?"

css = read_file(CSS_FILE)

# Check for CJK fonts
cjk_fonts = re.findall(r'(?i)(noto\s+sans\s+(cjk|sc|tc|jp|kr|hk)|source\s+han|hiragino|ming|gothic|sans-jp)', css)

# Check for Arabic fonts
arabic_fonts = re.findall(r'(?i)(noto\s+sans\s+arabic|noto\s+naskh|amiri|scheherazade|arabic)', css)

# Check for Devanagari fonts
devanagari_fonts = re.findall(r'(?i)(noto\s+sans\s+devanagari|mangal|kohinoor|devanagari)', css)

# Check for Thai fonts
thai_fonts = re.findall(r'(?i)(noto\s+sans\s+thai|sarabun|tahoma.*thai|thai)', css)

# Check font-face declarations
font_face_count = len(re.findall(r'@font-face', css))

# Check Google Fonts imports
google_fonts = []
for html_file in HTML_FILES:
    path = os.path.join(WEB_DIR, html_file)
    content = read_file(path)
    fonts = re.findall(r'fonts\.googleapis\.com[^"]*', content)
    google_fonts.extend(fonts)

print(f"  @font-face declarations: {font_face_count}")
print(f"  Google Fonts imports: {len(set(google_fonts))}")
print(f"  CJK font references: {len(cjk_fonts)}")
print(f"  Arabic font references: {len(arabic_fonts)}")
print(f"  Devanagari font references: {len(devanagari_fonts)}")
print(f"  Thai font references: {len(thai_fonts)}")
if google_fonts:
    for gf in set(google_fonts):
        print(f"    {gf}")

has_non_latin_fonts = len(cjk_fonts) > 0 or len(arabic_fonts) > 0 or len(devanagari_fonts) > 0
record(
    "non-latin-font-stacks",
    has_non_latin_fonts,
    f"CJK: {len(cjk_fonts)}, Arabic: {len(arabic_fonts)}, Devanagari: {len(devanagari_fonts)}, Thai: {len(thai_fonts)}. "
    f"{'Non-Latin scripts may render as system fallback or tofu' if not has_non_latin_fonts else 'Font coverage present'}"
)

In [ ]:
# === Probe 9: Date formatting -- hardcoded vs locale-aware ===
# Floor 1 (James): "Does the product use en-US date formatting (MM/DD/YYYY) hardcoded or locale-aware?"
# Floor 49 (Ambassador): "Is 03/04/2026 unambiguous in every locale (March 4 vs April 3)?"

# Check Python files for hardcoded date formats
py_files = find_py_files(SRC_DIR)
hardcoded_date_patterns = [
    r'strftime\(["\']%m/%d',        # MM/DD
    r'strftime\(["\']%d/%m',        # DD/MM (also hardcoded)
    r'strftime\(["\']%Y-%m-%d',     # ISO (OK but not locale-aware)
    r'MM/DD/YYYY',
    r'DD/MM/YYYY',
]

# Check JS files for hardcoded date formatting
js_date_patterns = [
    r'toLocaleDateString\(["\']en-US',    # Hardcoded locale
    r'toLocaleString\(["\']en-US',         # Hardcoded locale
    r'new Intl\.DateTimeFormat\(["\']en-US',
    r'MM/DD/YYYY',
    r'getMonth\(\).*getDate\(\)',            # Manual date construction
]

py_date_hits = []
for pf in py_files:
    content = read_file(pf)
    for pat in hardcoded_date_patterns:
        matches = re.findall(pat, content)
        if matches:
            py_date_hits.append((str(pf.relative_to(PROJECT_ROOT)), pat, len(matches)))

js_date_hits = []
for js_file in JS_FILES:
    path = os.path.join(WEB_DIR, js_file)
    content = read_file(path)
    for pat in js_date_patterns:
        matches = re.findall(pat, content)
        if matches:
            js_date_hits.append((js_file, pat, len(matches)))

# Check if ISO 8601 is used (timezone-aware, unambiguous)
iso_usage = 0
for pf in py_files:
    content = read_file(pf)
    iso_usage += len(re.findall(r'\.isoformat\(\)', content))

print(f"  Python hardcoded date patterns: {len(py_date_hits)}")
for (f, p, n) in py_date_hits[:5]:
    print(f"    {f}: {p} ({n}x)")
print(f"  JS hardcoded date patterns: {len(js_date_hits)}")
for (f, p, n) in js_date_hits[:5]:
    print(f"    {f}: {p} ({n}x)")
print(f"  Python .isoformat() usage: {iso_usage}")

record(
    "date-format-locale-aware",
    len(py_date_hits) == 0 and len(js_date_hits) == 0,
    f"Python hardcoded: {len(py_date_hits)}, JS hardcoded: {len(js_date_hits)}, ISO usage: {iso_usage}. "
    f"{'03/04/2026 is ambiguous in non-US locales' if py_date_hits or js_date_hits else 'No hardcoded date formats found'}"
)

In [ ]:
# === Probe 10: Currency symbol handling ===
# Floor 1 (James): "Is the currency symbol always $ or does it adapt based on user's region?"
# Floor 7 (Dewi/Indonesian): "Are currency displays showing Rp instead of $?"
# Floor 11 (Priya/Hindi): "Is the rupee symbol used instead of $ for currency?"

# Check for hardcoded $ in locale files (non-English)
dollar_in_locales = []
for loc in sorted(existing_files - {"en"}):
    data = read_json(os.path.join(LOCALES_DIR, f"{loc}.json"))
    content_str = json.dumps(data, ensure_ascii=False)
    # Find $ that is NOT part of ${variable} interpolation pattern
    dollar_refs = re.findall(r'\$(?!\{)\d', content_str)
    if dollar_refs:
        dollar_in_locales.append((loc, len(dollar_refs)))

# Check JS for hardcoded currency formatting
js_currency_hits = []
for js_file in JS_FILES:
    path = os.path.join(WEB_DIR, js_file)
    content = read_file(path)
    # Look for hardcoded USD references
    patterns = [
        r"'USD'",
        r'"USD"',
        r"currency:\s*['\"]USD['\"]" ,
        r"\$\{?\.toFixed",
    ]
    for pat in patterns:
        if re.search(pat, content):
            js_currency_hits.append((js_file, pat))

# Check Python for hardcoded $ formatting
py_currency_hits = []
for pf in py_files:
    content = read_file(pf)
    if re.search(r'f"\$|f\'\$|\.format.*\$|\$\{amount|\$\{cost', content):
        py_currency_hits.append(str(pf.relative_to(PROJECT_ROOT)))

print(f"  Non-English locales with hardcoded $: {len(dollar_in_locales)}")
for (loc, n) in dollar_in_locales:
    print(f"    {loc}: {n} occurrences")
print(f"  JS hardcoded USD: {len(js_currency_hits)}")
print(f"  Python hardcoded $: {len(py_currency_hits)}")

# English locale uses $ in celebration strings intentionally (${amount})
# Non-English locales should adapt currency to local symbol
record(
    "currency-locale-aware",
    len(dollar_in_locales) == 0 and len(js_currency_hits) == 0,
    f"Locales with $: {[l for l,_ in dollar_in_locales]}. JS USD: {len(js_currency_hits)}. "
    f"Py $: {len(py_currency_hits)}. "
    f"{'Currency is US-centric' if dollar_in_locales else 'No hardcoded currency symbols in non-English locales'}"
)

In [ ]:
# === Probe 11: Pluralization support ===
# Floor 15 (Kasia/Polish): "Does the product have a plural engine for complex plurals?"
# Floor 15: "Would 'You have {count} recipe' produce 'You have 5 recipe' without plural rules?"
# Polish needs 4 plural forms (1, 2-4, 5-21, 22+)

# Check i18n.py for pluralization
i18n_content = read_file(I18N_FILE)
has_plural_fn = bool(re.search(r'def.*plural|plural.*form|ngettext|ICU.*Message|\bplural\b', i18n_content, re.IGNORECASE))

# Check for plural patterns in locale files
# ICU MessageFormat uses {count, plural, one{...} other{...}}
en_content_str = json.dumps(en_data, ensure_ascii=False)
icu_plural = re.findall(r'\{\w+,\s*plural', en_content_str)
simple_count = re.findall(r'\{count\}|\{n\}|\{num\}', en_content_str)

# Check for count-related strings that might need pluralization
plural_candidates = []
for key in flatten_keys(en_data):
    val = en_data
    for part in key.split("."):
        if isinstance(val, dict):
            val = val.get(part, "")
    if isinstance(val, str) and re.search(r'\d+\s+(run|app|day|hour|task|recipe|item)', val):
        plural_candidates.append((key, val[:80]))

print(f"  i18n.py has plural function: {has_plural_fn}")
print(f"  ICU MessageFormat patterns: {len(icu_plural)}")
print(f"  Simple count interpolation: {len(simple_count)}")
print(f"  Strings with counts needing pluralization: {len(plural_candidates)}")
for (k, v) in plural_candidates[:5]:
    print(f"    {k}: '{v}'")

record(
    "pluralization-support",
    has_plural_fn or len(icu_plural) > 0,
    f"Plural engine: {has_plural_fn}. ICU patterns: {len(icu_plural)}. "
    f"Count strings: {len(simple_count)}. Candidates needing plurals: {len(plural_candidates)}. "
    f"{'Polish/Russian/Arabic plural rules would break' if not has_plural_fn else 'Plural engine present'}"
)

In [ ]:
# === Probe 12: String interpolation -- parameterized vs concatenation ===
# Floor 49 (Ambassador): "Are technical terms translated, explained, or preserved with tooltips?"
# Check: does the i18n system use parameterized strings (safe) or string concatenation (unsafe for RTL/reordering)

# Check i18n.py uses .format() for interpolation
i18n_uses_format = bool(re.search(r'value\.format\(', i18n_content))

# Check locale files for parameterized strings using {name} pattern
parameterized_count = 0
concat_suspect = 0

en_str = json.dumps(en_data, ensure_ascii=False)
parameterized_count = len(re.findall(r'\{\w+\}', en_str))

# Check JS files for string concatenation with i18n strings
js_concat_hits = []
for js_file in JS_FILES:
    path = os.path.join(WEB_DIR, js_file)
    content = read_file(path)
    # String concat with + that might indicate hardcoded joining
    concat_patterns = re.findall(r'"[^"]+"\s*\+\s*\w+\s*\+\s*"[^"]+"', content)
    if concat_patterns:
        js_concat_hits.extend([(js_file, p[:60]) for p in concat_patterns])

print(f"  i18n.py uses .format(): {i18n_uses_format}")
print(f"  Parameterized strings in en.json: {parameterized_count}")
print(f"  JS string concatenation suspects: {len(js_concat_hits)}")
for (f, p) in js_concat_hits[:5]:
    print(f"    {f}: {p}")

record(
    "string-interpolation-safe",
    i18n_uses_format and parameterized_count > 0,
    f"i18n .format(): {i18n_uses_format}. Parameterized: {parameterized_count}. "
    f"JS concat suspects: {len(js_concat_hits)}. "
    f"{'Parameterized interpolation present -- safe for word-order changes' if i18n_uses_format else 'Missing interpolation'}"
)

In [ ]:
# === Probe 13: Number formatting locale awareness ===
# Floor 7 (Dewi/Indonesian): "Does the locale correctly handle Indonesian number formatting (period as thousands separator)?"
# Floor 48 (Saint Solace): "Are number, date, and currency formats locale-aware (not hardcoded en-US)?"

# Check JS for Intl.NumberFormat or toLocaleString with hardcoded locale
js_number_hits = []
for js_file in JS_FILES:
    path = os.path.join(WEB_DIR, js_file)
    content = read_file(path)
    # Hardcoded number formatting
    patterns = [
        (r'toLocaleString\([\'"]en', 'toLocaleString(en-*)'),
        (r'Intl\.NumberFormat\([\'"]en', 'Intl.NumberFormat(en-*)'),
        (r'\.toFixed\(\d\)', '.toFixed()'),
    ]
    for pat, desc in patterns:
        matches = re.findall(pat, content)
        if matches:
            js_number_hits.append((js_file, desc, len(matches)))

# Check Python for hardcoded number formatting
py_number_hits = []
for pf in py_files:
    content = read_file(pf)
    if re.search(r'f"\{[^}]+:,\}|f"\{[^}]+:\.\df\}|"{:,}"', content):
        py_number_hits.append(str(pf.relative_to(PROJECT_ROOT)))

print(f"  JS hardcoded number formatting: {len(js_number_hits)}")
for (f, d, n) in js_number_hits:
    print(f"    {f}: {d} ({n}x)")
print(f"  Python hardcoded number formatting: {len(py_number_hits)}")

record(
    "number-format-locale-aware",
    len(js_number_hits) == 0,
    f"JS hardcoded: {len(js_number_hits)}, Python hardcoded: {len(py_number_hits)}. "
    f"{'1,000.50 vs 1.000,50 -- number format is locale-dependent' if js_number_hits else 'No hardcoded number formats found'}"
)

In [ ]:
# === Probe 14: Locale fallback chain and detect_locale() ===
# Floor 4 (Lucas/Portuguese): "Does the Portuguese locale serve pt-BR or pt-PT?"
# Floor 20 (Olena/Ukrainian): "Does the product have a locale inheritance/fallback system?"
# Floor 25 (Aisyah/Malay): "Does the product have a locale fallback chain (ms -> id)?"

# Parse detect_locale() from i18n.py
detect_fn = re.search(r'def detect_locale.*?(?=\ndef |\Z)', i18n_content, re.DOTALL)
detect_body = detect_fn.group(0) if detect_fn else ""

# Check locale mappings
locale_mappings = {}
map_matches = re.findall(r'"(\S+)":\s*"(\w+)"', detect_body)
for tag, target in map_matches:
    locale_mappings[tag] = target

print(f"  detect_locale() mapping entries: {len(locale_mappings)}")
print(f"  Key mappings:")

# Check specific sub-locale mappings that tower questions ask about
critical_mappings = {
    "pt-br": "pt",    # Floor 4: pt-BR vs pt-PT
    "zh-tw": "zh-hant",  # Floor 38: zh-TW must NOT map to zh
    "zh-cn": "zh",    # Floor 38: zh-CN -> zh (Simplified)
    "zh-hant": "zh-hant",  # Floor 38: zh-Hant -> zh-hant
}

mapping_issues = []
for tag, expected in critical_mappings.items():
    actual = locale_mappings.get(tag, "NOT MAPPED")
    status = "OK" if actual == expected else f"MISMATCH (got '{actual}', expected '{expected}')"
    print(f"    {tag} -> {actual} [{status}]")
    if actual != expected:
        mapping_issues.append((tag, expected, actual))

# Check: pt-PT is NOT mapped (no European Portuguese support)
pt_pt_mapped = "pt-pt" in locale_mappings
print(f"\n  pt-PT explicitly mapped: {pt_pt_mapped}")
print(f"  Malay (ms) fallback to Indonesian (id): {'ms' in locale_mappings}")

record(
    "locale-fallback-chain",
    len(mapping_issues) == 0 and len(locale_mappings) > 10,
    f"{len(locale_mappings)} mappings. Issues: {len(mapping_issues)}. "
    f"pt-PT: {'mapped' if pt_pt_mapped else 'NOT mapped (no European Portuguese)'}. "
    f"ms->id: {'yes' if 'ms' in locale_mappings else 'no fallback for Malay'}"
)

In [ ]:
# === Probe 15: Unicode encoding safety in Python ===
# Floor 8 (Linh/Vietnamese): "Do Vietnamese diacritics render correctly?"
# Floor 49 (Ambassador): "Can a user with a non-ASCII name see their name rendered correctly everywhere?"
# Check: do all Python files that read/write text use UTF-8 encoding explicitly?

encoding_issues = []
for pf in py_files:
    content = read_file(pf)
    # Find file opens without explicit encoding
    unsafe_opens = re.findall(r'open\([^)]+\)(?!.*encoding)', content)
    # Filter: only flag text mode opens (not 'rb'/'wb')
    for m in unsafe_opens:
        if "'rb'" not in m and '"rb"' not in m and "'wb'" not in m and '"wb"' not in m:
            if 'encoding' not in m:
                encoding_issues.append((str(pf.relative_to(PROJECT_ROOT)), m[:80]))

# Check i18n.py specifically loads JSON with utf-8
i18n_utf8 = bool(re.search(r'encoding="utf-8"', i18n_content))

# Check for Path.read_text() with encoding
read_text_safe = len(re.findall(r'read_text\(encoding="utf-8"\)', i18n_content))
read_text_unsafe = len(re.findall(r'read_text\(\)', i18n_content))

print(f"  i18n.py uses UTF-8 encoding: {i18n_utf8}")
print(f"  i18n.py read_text(encoding=\"utf-8\"): {read_text_safe}")
print(f"  i18n.py read_text() without encoding: {read_text_unsafe}")
print(f"  Python open() without encoding: {len(encoding_issues)}")
for (f, m) in encoding_issues[:5]:
    print(f"    {f}: {m}")

record(
    "unicode-encoding-safety",
    i18n_utf8 and read_text_unsafe == 0,
    f"i18n.py UTF-8: {i18n_utf8}. Unsafe read_text: {read_text_unsafe}. "
    f"Open without encoding: {len(encoding_issues)}. "
    f"{'Vietnamese/Arabic/CJK text will be correctly decoded' if i18n_utf8 else 'Encoding safety gap'}"
)

In [ ]:
# === Probe 16: i18n.py module architecture ===
# Floor 48 (Saint Solace): "Is there a community contribution path for translation?"
# Floor 49 (Ambassador): "In 2030 with 100 countries, does the i18n infrastructure scale?"
# Structural check: does i18n.py have the key functions needed?

required_functions = [
    ("t(", "Main translation function"),
    ("set_locale(", "Set active locale"),
    ("get_locale(", "Get active locale"),
    ("get_strings(", "Get full locale dict"),
    ("detect_locale(", "Auto-detect from Accept-Language"),
    ("is_rtl(", "Check if locale is RTL"),
    ("get_direction(", "Get text direction"),
    ("js_bundle(", "Generate JS i18n payload"),
    ("_RTL_LOCALES", "RTL locale set"),
    ("_SUPPORTED", "Supported locale set"),
]

present = []
missing_fns = []
for fn, desc in required_functions:
    if fn in i18n_content:
        present.append((fn, desc))
    else:
        missing_fns.append((fn, desc))

# Check for lru_cache (performance)
has_cache = "lru_cache" in i18n_content

# Check for English fallback
has_fallback = bool(re.search(r'fallback|_load\("en"\)', i18n_content))

print(f"  Functions present: {len(present)}/{len(required_functions)}")
for (fn, desc) in present:
    print(f"    {fn} -- {desc}")
if missing_fns:
    print(f"  MISSING:")
    for (fn, desc) in missing_fns:
        print(f"    {fn} -- {desc}")
print(f"  LRU cache: {has_cache}")
print(f"  English fallback: {has_fallback}")

record(
    "i18n-module-architecture",
    len(missing_fns) == 0 and has_cache and has_fallback,
    f"{len(present)}/{len(required_functions)} functions present. "
    f"Cache: {has_cache}. Fallback: {has_fallback}. "
    f"{'Architecture scales for 100+ locales' if len(missing_fns) == 0 else f'Missing: {[f for f,_ in missing_fns]}'}"
)

In [ ]:
# === Probe 17: US-centric cultural content ===
# Floor 48 (Saint Solace): "Is cultural context respected (holidays, calendars, customs not US-centric)?"
# Check holidays in en.json for US-centric bias

holidays_data = en_data.get("delight", {}).get("holidays", {})

# Categorize holidays
us_specific = []
universal = []
other_culture = []

us_holiday_names = {"Independence Day", "Thanksgiving", "4th of July"}
universal_names = {"New Year", "Valentine's Day", "Easter", "Mother's Day", "Father's Day",
                   "Christmas", "Halloween"}
non_western = {"Lunar New Year", "Diwali"}

for key, val in holidays_data.items():
    if key.endswith("_name"):
        name = val
        if any(us in name for us in us_holiday_names):
            us_specific.append(name)
        elif any(nw in name for nw in non_western):
            other_culture.append(name)
        else:
            universal.append(name)

# Check for Eid, Ramadan, Hanukkah, Chuseok, Obon, etc.
missing_global = ["Eid al-Fitr", "Ramadan", "Hanukkah", "Chuseok", "Obon", "Nowruz",
                  "Vesak", "Dragon Boat"]
holidays_str = json.dumps(holidays_data)
present_global = [h for h in missing_global if h.lower() in holidays_str.lower()]
absent_global = [h for h in missing_global if h.lower() not in holidays_str.lower()]

print(f"  US-specific holidays: {us_specific}")
print(f"  Universal holidays: {universal}")
print(f"  Non-Western holidays present: {other_culture}")
print(f"  Major global holidays absent: {absent_global}")

# Pass if at least SOME non-Western holidays exist
record(
    "cultural-content-diversity",
    len(other_culture) >= 2 and len(absent_global) < len(missing_global),
    f"US-only: {len(us_specific)}, Universal: {len(universal)}, Non-Western: {len(other_culture)}. "
    f"Absent global holidays: {absent_global}. "
    f"{'Some cultural diversity but missing major non-Western holidays' if other_culture else 'US-centric holiday calendar'}"
)

In [ ]:
# === EVIDENCE SUMMARY ===

passed = sum(1 for _, p, _ in results if p)
total = len(results)
score = passed / total * 100 if total else 0

print("=" * 70)
print(f"TOWER {TOWER}: {TOWER_NAME}")
print(f"NORTHSTAR: {NORTHSTAR}")
print(f"SCORE: {passed}/{total} probes passed ({score:.1f}%)")
print(f"VERDICT: {'PASS' if score >= MIN_SCORE else 'FAIL'} (threshold: {MIN_SCORE}%)")
print("=" * 70)
print()

for name, p, detail in results:
    tag = "PASS" if p else "FAIL"
    print(f"  [{tag}] {name}")
    if detail and not p:
        print(f"         {detail}")

print()

# Evidence JSON
evidence = {
    "tower": TOWER,
    "tower_name": TOWER_NAME,
    "northstar": NORTHSTAR,
    "notebook": NOTEBOOK_PATH,
    "total_probes": total,
    "passed": passed,
    "failed": total - passed,
    "score_pct": round(score, 1),
    "verdict": "PASS" if score >= MIN_SCORE else "FAIL",
    "min_score": MIN_SCORE,
    "probes": [
        {"name": name, "passed": p, "detail": detail}
        for name, p, detail in results
    ],
    "locale_coverage": {
        "expected": len(EXPECTED_LOCALES),
        "found": len(existing_files),
        "missing": missing_files,
    },
    "rtl_locales": sorted(RTL_LOCALES),
    "dna": "global(reach) = locale(correct) x culture(respected) x script(rendered) x format(native)",
}

print(json.dumps(evidence, indent=2))